# Other ice-dammed lakes: Sentinel-2 scenes for the transfer test

Outbursts of ice-dammed lakes from version 4.2 of the global GLOF database (Lützow et al., 2023): events from 2017 to 2025 dated to within 10 days, more than 30 km from Shisper, at most six per lake (`data/glofdb_*.csv`). For each outburst, Sentinel-2 L2A scenes from 60 days before to 60 days after are prepared as for Shisper: a 5 km window centred on the lake, bands B4, B3, B2, B8, B11 and B12 read at 10 m and resized to 128 x 128, at most 20 % cloud or cloud shadow and at least 95 % valid data. Scenes within 30 days after an outburst are labelled GLOF, and scenes in the uncertain period of any recorded outburst at the lake are left out.

**Output** (`RiskFlow_multisite_scenes`): one `.npz` per lake with the scenes, dates, labels, cloud fraction and open water within 1.5 km of the lake. CPU runtime, 1 to 2 hours; finished lakes are skipped on a rerun.

## Setup

In [ ]:
%pip install -q pystac-client planetary-computer rasterio

import os, time, threading
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, tensorflow as tf
import pystac_client, planetary_computer, rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from pyproj import Transformer
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/RiskFlow_multisite_scenes'
REPO = 'https://raw.githubusercontent.com/ZuhaFatima96/IceWatch-A-Multimodal-Deep-Learning-Framework-for-Early-Warning-of-GLOFs/main'
DATA = '../data' if os.path.isdir('../data') else f'{REPO}/data'
os.makedirs(OUT, exist_ok=True)

WINDOW_KM, IMG, NPX = 5.0, 128, 500         # 5 km at 10 m, resized to 128 x 128
BEFORE, AFTER, GLOF_DAYS = 60, 60, 30
MAX_CLOUD, MIN_VALID = 0.20, 0.95
CLOUD = [3, 8, 9, 10]                       # SCL: cloud shadow, cloud medium and high probability, thin cirrus
BANDS = ['B04', 'B03', 'B02', 'B08', 'B11', 'B12']
NEAR_KM = 1.5

LAKES = pd.read_csv(f'{DATA}/glofdb_lakes.csv')                                     # lakes used
EVENTS = pd.read_csv(f'{DATA}/glofdb_outbursts.csv', parse_dates=['lo', 'hi'])     # outbursts used
ALLEV = pd.read_csv(f'{DATA}/glofdb_outbursts_all.csv', parse_dates=['lo', 'hi'])  # every recorded outburst at these lakes
print(f'{len(LAKES)} lakes | {len(EVENTS)} outbursts')
print(LAKES.region.value_counts().to_string())

def open_catalog():
    for wait in [30, 60, 120, 240, 300, 0]:        # the Planetary Computer limits request rates
        try:
            return pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')
        except Exception:
            if not wait:
                raise
            time.sleep(wait)
catalog = open_catalog()

## Helpers

In [ ]:
DAY = pd.Timedelta(days=1)

def label_of(t, lake_events):
    # 1: within 30 days after a precisely dated outburst; None: uncertain (left out); 0: no outburst
    for r in lake_events[lake_events.precise].itertuples():
        if r.hi <= t <= r.lo + GLOF_DAYS * DAY:
            return 1, f'{r.lake_id}_{r.lo:%Y%m%d}'
    for r in lake_events.itertuples():
        if r.lo <= t <= r.hi + GLOF_DAYS * DAY:
            return None, ''
    return 0, ''

def epsg_of(it):
    p = it.properties
    for k in ('proj:epsg', 'proj:code'):
        if p.get(k):
            return int(str(p[k]).split(':')[-1])
    t = str(p.get('s2:mgrs_tile', ''))
    zone, band = int(t[:2]), t[2:3].upper()
    return (32600 if band >= 'N' else 32700) + zone

_TR = {}
def bounds_for(epsg, lat, lon):
    if epsg not in _TR:
        _TR[epsg] = Transformer.from_crs('EPSG:4326', f'EPSG:{epsg}', always_xy=True)
    x, y = _TR[epsg].transform(lon, lat)
    h = WINDOW_KM * 500.0
    return (x - h, y - h, x + h, y + h)

STATS, _LOCK = dict(reads=0, failed=0), threading.Lock()
def read(href, bounds, n):
    for k in range(5):
        try:
            url = planetary_computer.sign(href)    # sign just before reading: tokens expire
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_MAX_RETRY='5'):
                with rasterio.open(url) as src:
                    w = from_bounds(*bounds, transform=src.transform)
                    a = src.read(1, window=w, out_shape=(n, n), boundless=True, fill_value=0,
                                 resampling=Resampling.nearest)
            with _LOCK:
                STATS['reads'] += 1
            return a
        except Exception:
            try:
                planetary_computer.sas.TOKEN_CACHE.clear()
            except Exception:
                pass
            if k == 4:
                with _LOCK:
                    STATS['reads'] += 1; STATS['failed'] += 1
                return None
            time.sleep(10 * (k + 1))

def check_reads():
    # never save an incomplete lake; it is retried on the next run
    if STATS['failed'] > 0.02 * max(STATS['reads'], 1):
        raise RuntimeError(f"{STATS['failed']} of {STATS['reads']} reads failed")

def mosaic(arrs):
    out = None
    for a in arrs:
        if a is not None:
            out = a.copy() if out is None else np.where(out == 0, a, out)
    return out

def baseline(it):
    try:
        return float(it.properties.get('s2:processing_baseline', 0))
    except (TypeError, ValueError):
        return 0.0

yy, xx = np.mgrid[0:NPX, 0:NPX]
NEAR = np.hypot(yy - (NPX - 1) / 2, xx - (NPX - 1) / 2) * (WINDOW_KM / NPX) <= NEAR_KM
def water_km2(stack):
    # same water test as for Shisper; stack (6, H, W) in BANDS order
    g, nir, sw = (stack[i].astype('float32') for i in (1, 3, 4))
    ndwi = (g - nir) / (g + nir + 1e-6)
    return float(((ndwi > 0.15) & (nir < 1500) & (sw < 1000) & (g > 0) & NEAR).sum()) * (WINDOW_KM * 1000 / NPX) ** 2 / 1e6

## Export, lake by lake

In [ ]:
def do_lake(lk):
    ev = EVENTS[EVENTS.lake_id == lk.lake_id]
    allev = ALLEV[ALLEV.lake_id == lk.lake_id]
    wins = sorted((r.lo - BEFORE * DAY, r.hi + AFTER * DAY) for r in ev.itertuples())
    merged = []
    for a, b in wins:
        if merged and a <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], b))
        else:
            merged.append((a, b))
    bbox = [lk.lon - 0.02, lk.lat - 0.02, lk.lon + 0.02, lk.lat + 0.02]
    items = {}
    STATS.update(reads=0, failed=0)
    for a, b in merged:
        for k in range(6):
            try:
                for it in catalog.search(collections=['sentinel-2-l2a'], bbox=bbox,
                                         datetime=f'{a:%Y-%m-%d}/{b:%Y-%m-%d}', limit=1000).items():
                    items[it.id] = it
                break
            except Exception:
                if k == 5:
                    raise
                time.sleep(30 * (k + 1))
    items = list(items.values())
    if not items:
        return dict(items=0)
    ep = pd.Series([epsg_of(it) for it in items]).mode().iloc[0]      # one projection per lake, one mosaic per day
    items = [it for it in items if epsg_of(it) == ep]
    B = bounds_for(ep, lk.lat, lk.lon)
    days = {}
    for it in items:
        days.setdefault(it.datetime.strftime('%Y-%m-%d'), []).append(it)
    with ThreadPoolExecutor(12) as pool:
        scl = dict(zip(list(days), pool.map(lambda d: mosaic([read(it.assets['SCL'].href, B, NPX // 2) for it in days[d]]),
                                            list(days))))
    check_reads()
    rows = []
    for d, m in scl.items():
        if m is None:
            continue
        valid = m > 0
        cloud = float(np.isin(m, CLOUD).sum()) / max(int(valid.sum()), 1)
        lab, eid = label_of(pd.Timestamp(d), allev)
        if valid.mean() >= MIN_VALID and cloud <= MAX_CLOUD and lab is not None:
            rows.append(dict(date=d, cloud=round(cloud, 3), label=lab, event_id=eid))
    n_amb = sum(label_of(pd.Timestamp(d), allev)[0] is None for d in scl)
    if not rows:
        return dict(items=len(items), days=len(days), clear=0, ambiguous=n_amb)

    tasks = [(r['date'], j, b) for r in rows for j, it in enumerate(days[r['date']])
             if all(bb in it.assets for bb in BANDS) for b in BANDS]
    with ThreadPoolExecutor(12) as pool:
        got = dict(zip(tasks, pool.map(lambda t: read(days[t[0]][t[1]].assets[t[2]].href, B, NPX), tasks)))
    def load_day(d):
        stacks = []
        for j, it in enumerate(days[d]):
            if not all(b in it.assets for b in BANDS):
                continue
            bands = [got[(d, j, b)] for b in BANDS]
            if any(x is None for x in bands):
                continue
            st = np.stack(bands)
            if baseline(it) >= 4.0:            # remove the +1000 offset of processing baseline 04.00 and later
                st = np.where(st > 0, np.clip(st.astype('int32') - 1000, 1, None), 0).astype('uint16')
            stacks.append(st)
        return mosaic(stacks)
    stacks = [load_day(r['date']) for r in rows]
    check_reads()
    X, keep = [], []
    for r, st in zip(rows, stacks):
        if st is None or (st[0] > 0).mean() < MIN_VALID:
            continue
        r['water_km2'] = round(water_km2(st), 4)
        small = tf.image.resize(np.transpose(st, (1, 2, 0)).astype('float32'), [IMG, IMG], antialias=True).numpy()
        X.append(np.clip(np.rint(small), 0, 65535).astype('uint16'))
        keep.append(r)
    if not keep:
        return dict(items=len(items), days=len(days), clear=0, ambiguous=n_amb)
    meta = pd.DataFrame(keep)
    np.savez_compressed(f'{OUT}/{lk.lake_id}.npz', X=np.stack(X), date=meta.date.values, label=meta.label.values,
                        event_id=meta.event_id.values.astype(str), water_km2=meta.water_km2.values,
                        cloud=meta.cloud.values, epsg=ep, v=2)
    return dict(items=len(items), days=len(days), clear=len(meta), glof=int(meta.label.sum()), ambiguous=n_amb)

t0, LOG = time.time(), []
for i, lk in enumerate(LAKES.itertuples()):
    f = f'{OUT}/{lk.lake_id}.npz'
    if os.path.exists(f):
        try:
            if 'v' in np.load(f).files:
                continue
        except Exception:
            pass
    try:
        r = do_lake(lk)
    except Exception as e:
        r = dict(error=str(e)[:150])
        time.sleep(120)                          # let the rate limit recover before the next lake
    if r.get('items', 1) == 0 or r.get('clear', 1) == 0:
        # nothing usable: save an empty file so the lake is skipped next time
        np.savez_compressed(f, X=np.zeros((0, IMG, IMG, 6), 'uint16'), date=np.array([], str), label=np.array([], int),
                            event_id=np.array([], str), water_km2=np.array([]), cloud=np.array([]), epsg=0, v=2)
    LOG.append(dict(lake_id=lk.lake_id, **r))
    print(f'{i + 1:3d}/{len(LAKES)} {lk.region:22s} {str(lk.glacier)[:24]:24s} {r} | {(time.time() - t0) / 60:.1f} min', flush=True)
pd.DataFrame(LOG).to_csv(f'{OUT}/export_log_{int(time.time())}.csv', index=False)

## Summary

In [ ]:
metas, Xs = [], []
for lk in LAKES.itertuples():
    f = f'{OUT}/{lk.lake_id}.npz'
    if not os.path.exists(f):
        continue
    z = np.load(f, allow_pickle=True)
    if len(z['date']) == 0:
        continue
    m = pd.DataFrame(dict(date=z['date'], label=z['label'], event_id=z['event_id'], water_km2=z['water_km2'], cloud=z['cloud']))
    m['lake_id'], m['region'], m['k'] = lk.lake_id, lk.region, np.arange(len(m))
    metas.append(m); Xs.append(z['X'])
M = pd.concat(metas, ignore_index=True)
print(f'lakes with scenes: {M.lake_id.nunique()} of {len(LAKES)} | scenes: {len(M)} | GLOF scenes: {int(M.label.sum())} '
      f'from {M[M.label == 1].event_id.nunique()} outbursts')
print(M.groupby('region').agg(lakes=('lake_id', 'nunique'), scenes=('label', 'size'), glof=('label', 'sum')).to_string())

# was the lake visible before its outburst? (at least 0.02 km² of water in a scene of the preceding 60 days)
vis = []
for r in EVENTS[EVENTS.lake_id.isin(M.lake_id)].itertuples():
    pre = M[(M.lake_id == r.lake_id) & (pd.to_datetime(M.date) < r.lo) & (pd.to_datetime(M.date) >= r.lo - BEFORE * DAY)]
    vis.append(dict(event_id=f'{r.lake_id}_{r.lo:%Y%m%d}', pre_scenes=len(pre),
                    max_water_before=pre.water_km2.max() if len(pre) else np.nan))
V = pd.DataFrame(vis)
V['visible'] = V.max_water_before >= 0.02
V.to_csv(f'{OUT}/event_visibility.csv', index=False)
print(f'outbursts with a clear scene before: {int((V.pre_scenes > 0).sum())} of {len(V)} | lake visible before: {int(V.visible.sum())}')

def pick(df, n):
    df = df.groupby('lake_id').head(1)
    return df.sample(min(n, len(df)), random_state=0)
idx = {lk: j for j, lk in enumerate(M.lake_id.unique())}
A = pick(M[M.label == 1], 12)
Bv = pick(M[(M.label == 0) & (M.water_km2 >= 0.02)], 12)
fig, ax = plt.subplots(4, 6, figsize=(18, 13))
panels = [(k, r) for k, (_, r) in enumerate(A.iterrows())] + [(12 + k, r) for k, (_, r) in enumerate(Bv.iterrows())]
for k, r in panels:
    a = ax[k // 6, k % 6]
    x = Xs[idx[r.lake_id]][r.k][..., [0, 1, 2]].astype('float32')
    lo, hi = np.percentile(x, 2), np.percentile(x, 98)
    a.imshow(np.clip((x - lo) / (hi - lo + 1e-6), 0, 1))
    a.set_title(f"{'GLOF' if r.label == 1 else 'lake before'} | {r.region[3:]}\n{r.date} | water {r.water_km2:.2f} km²", fontsize=9)
    a.set_xticks([]); a.set_yticks([])
for k in set(range(24)) - {k for k, _ in panels}:
    ax[k // 6, k % 6].axis('off')
plt.suptitle('Top: GLOF scenes. Bottom: scenes with a lake before an outburst.')
plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig(f'{OUT}/sample_scenes.png', dpi=80); plt.show()